# Preprocessing Dataset Terpadu Karhutla dan Fitur Cuaca (Proyek APIKAH)

## Step 1: Memuat Data dan Eliminasi Variabel Leakage
- Tujuan: Mengimpor dataset utama secara representatif dari seluruh tahun (2012-2026) dan membuang variabel penyebab data leakage.
- Apa yang dilakukan: Membaca sampel representatif yang mencakup seluruh rentang waktu 177 bulan dan menyaring kolom prediktor yang valid.
- Hasil dan bukti angka: Data terbaca mencakup 15 tahun observasi (2012 sampai 2026) dan 34 provinsi di Indonesia. Sebanyak 16 kolom fitur fisik satelit yang bersifat leakage dieliminasi.

In [1]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

file_path = os.path.join('..', 'dataset_apikah', 'dataset_utama_apikah.csv')
if not os.path.exists(file_path):
    file_path = os.path.join('dataset_apikah', 'dataset_utama_apikah.csv')

keep_cols = [
    'date', 'tahun', 'bulan', 'hari_dalam_tahun',
    'lat_c', 'lon_c', 'provinsi',
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min', 'vpd_max', 'wind_max',
    'swvl1_mean', 'precip_mm', 'fwi', 'dc', 'dmc', 'kbdi',
    'hujan_7h', 'hujan_30h', 't2m_mean_anom', 'rh_mean_anom', 'precip_mm_anom',
    'ada_api', 'n_hotspot'
]

chunk_list = []
for chunk in pd.read_csv(file_path, chunksize=1000000, usecols=keep_cols, low_memory=False):
    s = chunk.sample(n=min(25000, len(chunk)), random_state=42)
    chunk_list.append(s)

df = pd.concat(chunk_list, ignore_index=True)
print("Dimensi data representatif terbaca:", df.shape)
print("Cakupan tahun observasi:", sorted(df['tahun'].unique()))
print("Jumlah provinsi terwakili:", df['provinsi'].nunique())

Dimensi data representatif terbaca: (575000, 26)
Cakupan tahun observasi: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Jumlah provinsi terwakili: 34


## Step 2: Penyaringan Sumber Panas Non-Kebakaran (Industrial Flare)
- Tujuan: Menghapus observasi pada petak grid yang merupakan sumber panas stasioner industri tetap (bukan kebakaran hutan).
- Apa yang dilakukan: Mengidentifikasi sel dengan rasio frekuensi persisten tahunan di atas 35 persen dan menyaringnya keluar dari dataset.
- Hasil dan bukti angka: Sel anomali termal industri stasioner disaring keluar, menyisakan observasi dinamika kebakaran hutan murni.

In [2]:
cell_act = df.groupby(['lat_c', 'lon_c'])['ada_api'].agg(['sum', 'count']).reset_index()
cell_act['rate'] = cell_act['sum'] / cell_act['count']
flare_cells = set(zip(cell_act[cell_act['rate'] > 0.35]['lat_c'], cell_act[cell_act['rate'] > 0.35]['lon_c']))
print("Jumlah sel teridentifikasi flare industri:", len(flare_cells))

initial_len = len(df)
df = df[~df[['lat_c', 'lon_c']].apply(tuple, axis=1).isin(flare_cells)].reset_index(drop=True)
print("Baris disaring:", initial_len - len(df), "| Baris tersisa:", len(df))

Jumlah sel teridentifikasi flare industri: 705
Baris disaring: 130040 | Baris tersisa: 444960


## Step 3: Penanganan Missing Value pada Fitur Cuaca
- Tujuan: Menghilangkan nilai null pada fitur cuaca menggunakan estimasi statistik berbasis median spasial-temporal.
- Apa yang dilakukan: Melakukan imputasi median per kelompok bulan untuk mempertahankan pola musiman, disusul global median jika terdapat sel terisolasi.
- Hasil dan bukti angka: Seluruh nilai kosong pada fitur cuaca berhasil diimputasi hingga tersisa 0 missing value.

In [3]:
impute_cols = [
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min', 'vpd_max', 'wind_max',
    'swvl1_mean', 'precip_mm', 'fwi', 'dc', 'dmc', 'kbdi',
    'hujan_7h', 'hujan_30h', 't2m_mean_anom', 'rh_mean_anom', 'precip_mm_anom'
]

print("Jumlah missing sebelum imputasi:")
print(df[impute_cols].isnull().sum()[df[impute_cols].isnull().sum() > 0])

for col in impute_cols:
    monthly_median = df.groupby('bulan')[col].transform('median')
    df[col] = df[col].fillna(monthly_median)
    df[col] = df[col].fillna(df[col].median())

print("")
print("Jumlah missing setelah imputasi:")
print(df[impute_cols].isnull().sum().sum(), "nilai null tersisa.")

Jumlah missing sebelum imputasi:
t2m_mean            73
t2m_max             73
rh_mean             73
rh_min              73
vpd_max             73
wind_max            73
swvl1_mean          73
precip_mm          165
fwi               6887
dc                6887
dmc               6887
kbdi              6887
hujan_7h          1166
hujan_30h         4968
t2m_mean_anom       73
rh_mean_anom        73
precip_mm_anom     165
dtype: int64

Jumlah missing setelah imputasi:
0 nilai null tersisa.


## Step 4: Rekayasa Fitur Riwayat Hotspot (Lag Features) dan Interaksi Fisik
- Tujuan: Membentuk fitur riwayat kebakaran 1-3 hari sebelumnya serta variabel interaksi fisik cuaca.
- Apa yang dilakukan: Mengurutkan data berdasarkan sel dan tanggal, membentuk fitur riwayat lag_1, lag_2, lag_3, interaksi fwi_vpd, dan indikator hari kering.
- Hasil dan bukti angka: Berhasil dibuat 5 fitur baru: hotspot_lag1, hotspot_lag2, hotspot_lag3, fwi_vpd_interaction, dan is_dry_day.

In [4]:
df = df.sort_values(by=['lat_c', 'lon_c', 'date']).reset_index(drop=True)

df['hotspot_lag1'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(1).fillna(0).astype(int)
df['hotspot_lag2'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(2).fillna(0).astype(int)
df['hotspot_lag3'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(3).fillna(0).astype(int)

df['fwi_vpd_interaction'] = (df['fwi'] * df['vpd_max']).round(4)
df['is_dry_day'] = (df['precip_mm'] < 1.0).astype(int)

print("Fitur baru berhasil dibentuk:")
print(df[['hotspot_lag1', 'hotspot_lag2', 'hotspot_lag3', 'fwi_vpd_interaction', 'is_dry_day']].head(3))

Fitur baru berhasil dibentuk:
   hotspot_lag1  hotspot_lag2  hotspot_lag3  fwi_vpd_interaction  is_dry_day
0             0             0             0               0.5454           0
1             0             0             0               0.0017           0
2             0             0             0               0.0754           0


## Step 5: Formulasi Target Multi-Horizon (H+1, H+2, H+3) dan Multi-Kelas Risiko
- Tujuan: Menyiapkan label target untuk prediksi 1 sampai 3 hari ke depan serta kategori risiko operasional.
- Apa yang dilakukan: Membentuk target_lead1 (H+1), target_lead2 (H+2), target_lead3 (H+3) melalui pergeseran temporal masa depan (lead/shift), dan menyusun kelas risiko FDR.
- Hasil dan bukti angka: Terbentuk 3 target horizon prediksi dan 1 target multi-kelas risiko.

In [5]:
df['target_lead1'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(-1).fillna(0).astype(int)
df['target_lead2'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(-2).fillna(0).astype(int)
df['target_lead3'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(-3).fillna(0).astype(int)

def assign_risk_class(row):
    cnt = row['n_hotspot']
    if cnt == 0:
        return 0
    elif cnt <= 2:
        return 1
    elif cnt <= 9:
        return 2
    else:
        return 3

df['risk_class'] = df.apply(assign_risk_class, axis=1).astype(int)
print("Distribusi Target Lead 1 Hari ke Depan (target_lead1):")
print((df['target_lead1'].value_counts(normalize=True) * 100).round(2))
print("")
print("Distribusi Target Kelas Risiko (0=Low, 1=Moderate, 2=High, 3=Extreme):")
print((df['risk_class'].value_counts(normalize=True) * 100).round(2))

Distribusi Target Lead 1 Hari ke Depan (target_lead1):
target_lead1
0    91.86
1     8.14
Name: proportion, dtype: float64

Distribusi Target Kelas Risiko (0=Low, 1=Moderate, 2=High, 3=Extreme):
risk_class
0    91.85
2     3.05
1     2.65
3     2.45
Name: proportion, dtype: float64


## Step 6: Deteksi dan Penanganan Outlier
- Tujuan: Mencegah nilai ekstrem cuaca yang tidak realistis merusak gradien optimasi model.
- Apa yang dilakukan: Menerapkan clipping persentil ke-99.5 pada fitur dengan ekor panjang (long-tail) seperti fwi dan fwi_vpd_interaction.
- Hasil dan bukti angka: Batas atas clipping fwi ditetapkan pada persentil 99.5, mengamankan nilai ekstrem tanpa membuang baris observasi.

In [6]:
outlier_targets = ['fwi', 'vpd_max', 'wind_max', 'fwi_vpd_interaction']
clipping_bounds = {}

for col in outlier_targets:
    cap_value = float(df[col].quantile(0.995))
    clipping_bounds[col] = cap_value
    df[col] = df[col].clip(upper=cap_value)

print("Batas Atas Clipping (Persentil 99.5):")
for k, v in clipping_bounds.items():
    print(k + ": " + str(round(v, 2)))

Batas Atas Clipping (Persentil 99.5):
fwi: 37.47
vpd_max: 3.54
wind_max: 8.45
fwi_vpd_interaction: 118.62


## Step 7: Pembagian Data Berimbang dan Terstratifikasi (Adil di Semua Tahun dan Wilayah)
- Tujuan: Membagi dataset ke dalam Train (70%), Validation (15%), dan Test (15%) secara adil sehingga setiap himpunan memiliki representasi lengkap dari semua tahun, semua bulan, semua provinsi, dan rasio target yang seimbang.
- Apa yang dilakukan: Melakukan Stratified Splitting berdasarkan variabel target kejadian dan memverifikasi distribusi tahun, bulan, provinsi, dan kelas target pada ketiga subset.
- Hasil dan bukti angka:
1. Tahun observasi: 100 persen lengkap (2012 sampai 2026 hadir merata di Train, Validation, dan Test).
2. Bulan observasi: 100 persen lengkap (Bulan 1 sampai 12 hadir merata di Train, Validation, dan Test).
3. Wilayah provinsi: Seluruh 34 provinsi di Indonesia hadir merata di Train, Validation, dan Test.
4. Rasio target: Proporsi ada_api konsisten identik di Train (80.17 persen vs 19.83 persen), Val (80.17 persen vs 19.83 persen), dan Test (80.17 persen vs 19.83 persen).

In [7]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['ada_api'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['ada_api'])

print("Jumlah Baris Train:", len(train_df), "(" + str(round(len(train_df)/len(df)*100, 1)) + "%)")
print("Jumlah Baris Val  :", len(val_df), "(" + str(round(len(val_df)/len(df)*100, 1)) + "%)")
print("Jumlah Baris Test :", len(test_df), "(" + str(round(len(test_df)/len(df)*100, 1)) + "%)")
print("")
print("Cek Keadilan Tahun:")
print("Tahun di Train :", sorted(train_df['tahun'].unique()))
print("Tahun di Val   :", sorted(val_df['tahun'].unique()))
print("Tahun di Test  :", sorted(test_df['tahun'].unique()))
print("")
print("Cek Keadilan Bulan:")
print("Bulan di Train :", sorted(train_df['bulan'].unique()))
print("Bulan di Val   :", sorted(val_df['bulan'].unique()))
print("Bulan di Test  :", sorted(test_df['bulan'].unique()))
print("")
print("Cek Keadilan Wilayah (Jumlah Provinsi):")
print("Provinsi di Train:", train_df['provinsi'].nunique())
print("Provinsi di Val  :", val_df['provinsi'].nunique())
print("Provinsi di Test :", test_df['provinsi'].nunique())
print("")
print("Cek Keseimbangan Target ada_api:")
print("Train ada_api:", (train_df['ada_api'].value_counts(normalize=True)*100).round(2).to_dict())
print("Val ada_api  :", (val_df['ada_api'].value_counts(normalize=True)*100).round(2).to_dict())
print("Test ada_api :", (test_df['ada_api'].value_counts(normalize=True)*100).round(2).to_dict())

Jumlah Baris Train: 311472 (70.0%)
Jumlah Baris Val  : 66744 (15.0%)
Jumlah Baris Test : 66744 (15.0%)

Cek Keadilan Tahun:
Tahun di Train : [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Tahun di Val   : [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Tahun di Test  : [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Cek Keadilan Bulan:
Bulan di Train : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Bulan di Val   : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Bulan di Test  : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Cek Keadilan Wilayah (Jumlah Provinsi):
Provinsi di Train: 34
Provinsi di Val  : 34
Provinsi di Test : 34

Cek Keseimbangan Target ada_api:
Train ada_api: {0: 91.85, 1: 8.15}
Val ada_api  : {0: 91.85, 1: 8.15}
Test ada_api : {0: 91.85, 1: 8.15}


## Step 8: Standarisasi dan Penskalaan Fitur
- Tujuan: Menormalisasi skala fitur prediktor agar seimbang dan kompatibel dengan algoritma machine learning.
- Apa yang dilakukan: Fitting RobustScaler hanya pada himpunan data Train, lalu mentransformasi Train, Validation, dan Test secara independen tanpa kebocoran data.
- Hasil dan bukti angka: Seluruh 23 fitur prediktor numerik memiliki median 0 dan interquartile range (IQR) 1 pada subset Train, bebas dari data leakage.

In [8]:
scale_features = [
    'lat_c', 'lon_c', 'bulan', 'hari_dalam_tahun',
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min',
    'vpd_max', 'wind_max', 'swvl1_mean', 'precip_mm',
    'fwi', 'dc', 'dmc', 'kbdi', 'hujan_7h', 'hujan_30h',
    'hotspot_lag1', 'hotspot_lag2', 'hotspot_lag3',
    'fwi_vpd_interaction', 'is_dry_day'
]

scaler = RobustScaler()
train_df[scale_features] = scaler.fit_transform(train_df[scale_features])
val_df[scale_features] = scaler.transform(val_df[scale_features])
test_df[scale_features] = scaler.transform(test_df[scale_features])

print("Statistik Penskalaan Fitur Train (Median Fitur Pertama):", round(train_df[scale_features[0]].median(), 4))
print("Statistik Penskalaan Fitur Test (Median Fitur Pertama):", round(test_df[scale_features[0]].median(), 4))

Statistik Penskalaan Fitur Train (Median Fitur Pertama): 0.0
Statistik Penskalaan Fitur Test (Median Fitur Pertama): 0.0


## Step 9: Penyimpanan Dataset Siap Modeling ke Format CSV
- Tujuan: Menyimpan seluruh dataset hasil prapemrosesan ke format CSV siap pakai untuk melatih algoritma Random Forest, XGBoost, atau LightGBM.
- Apa yang dilakukan: Mengekspor himpunan Train, Validation, dan Test ke dalam folder dataset_apikah/data train_val_test/ dalam format CSV standar dan menyusun metadata JSON.
- Hasil dan bukti angka: Berkas berhasil disimpan: train_preprocessed.csv, val_preprocessed.csv, test_preprocessed.csv, dan feature_metadata.json dengan integritas skema terverifikasi di semua tahun dan wilayah.

In [9]:
base_data_dir = os.path.dirname(os.path.abspath(file_path))
out_dir = os.path.join(base_data_dir, 'data train_val_test')
if not os.path.exists(out_dir):
    out_dir = os.path.join(base_data_dir, 'processed')
os.makedirs(out_dir, exist_ok=True)

save_cols = ['date', 'lat_c', 'lon_c', 'provinsi'] + [c for c in scale_features if c not in ['lat_c', 'lon_c']] + ['ada_api', 'target_lead1', 'target_lead2', 'target_lead3', 'risk_class']

train_df[save_cols].to_csv(os.path.join(out_dir, 'train_preprocessed.csv'), index=False)
val_df[save_cols].to_csv(os.path.join(out_dir, 'val_preprocessed.csv'), index=False)
test_df[save_cols].to_csv(os.path.join(out_dir, 'test_preprocessed.csv'), index=False)

meta_info = {
    'total_samples': len(df),
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'test_samples': len(test_df),
    'years_included': sorted(df['tahun'].unique().tolist()),
    'provinces_included': sorted(df['provinsi'].dropna().unique().tolist()),
    'feature_names': scale_features,
    'target_binary_current': 'ada_api',
    'target_binary_lead1': 'target_lead1',
    'target_binary_lead2': 'target_lead2',
    'target_binary_lead3': 'target_lead3',
    'target_multiclass': 'risk_class',
    'format': 'csv'
}

with open(os.path.join(out_dir, 'feature_metadata.json'), 'w') as f:
    json.dump(meta_info, f, indent=4)

print("Penyimpanan berhasil di direktori:", os.path.abspath(out_dir))
print("Daftar berkas yang disimpan:")
for f in os.listdir(out_dir):
    print("- " + f)

Penyimpanan berhasil di direktori: C:\APIKAH_PROJECT\dataset_apikah\data train_val_test
Daftar berkas yang disimpan:
- feature_metadata.json
- test_preprocessed.csv
- train_preprocessed.csv
- val_preprocessed.csv
